# Stage 4 - Vocals: RVC + FX (Kaggle GPU)

lyrics -> Piper TTS -> RVC voice conversion (blended vocalists) -> FX chain -> vocal stem.

**Before running:** GPU T4 x2 + **Internet ON** (Settings) so RVC/Piper/weights can download.

**Read** `docs/VOCALS_GUIDE.md` for the legal note and the honest scream limitation.

In [ ]:
import sys, os
from pathlib import Path
REPO_DIR = '/kaggle/working/metalcore'
if not Path(REPO_DIR).exists():
    !git clone https://github.com/YOUR_USERNAME/metalcore.git {REPO_DIR}
sys.path.insert(0, REPO_DIR); os.chdir(REPO_DIR)
!pip install -q -r requirements-vocals.txt

In [ ]:
# 1) One-time: clone RVC-Project + download pretrained weights, then install its deps.
!python -m rvc_training.cli setup
!pip install -q -r /kaggle/working/Retrieval-based-Voice-Conversion-WebUI/requirements.txt

In [ ]:
# 2) Build the vocal dataset. Organise isolated vocals per vocalist first:
#    data/vocals_raw/<vocalist>/*.wav  (use 'isolate' if you only have full mixes)
VOCALS_RAW = '/kaggle/input/YOUR_VOCALS'      # per-vocalist subfolders
RVC_DATA   = '/kaggle/working/rvc_dataset'
# Optional isolation from full mixes:
# !python -m rvc_training.cli isolate --input /kaggle/input/YOUR_REFS --output /kaggle/working/vocals_raw
!python -m rvc_training.cli prepare --input {VOCALS_RAW} --output {RVC_DATA}

In [ ]:
# 3) Train the blended voice (blend_mode: merged in configs/rvc.yaml).
!python -m rvc_training.cli train --dataset {RVC_DATA}/merged --name blend

In [ ]:
# 4) Download a Piper voice and point configs/rvc.yaml::piper_voice at it.
#    (Example voice; browse https://github.com/rhasspy/piper/blob/master/VOICES.md)
!mkdir -p /kaggle/working/piper_voices
%cd /kaggle/working/piper_voices
!wget -q https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/ljspeech/high/en_US-ljspeech-high.onnx
!wget -q https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/ljspeech/high/en_US-ljspeech-high.onnx.json
%cd {REPO_DIR}

In [ ]:
# 5) Generate a vocal stem from lyrics (from Stage 3).
import glob
MODEL = glob.glob('/kaggle/working/Retrieval-based-Voice-Conversion-WebUI/logs/blend/*.pth')[0]
INDEX = glob.glob('/kaggle/working/Retrieval-based-Voice-Conversion-WebUI/logs/blend/added_*.index')[0]
print('model:', MODEL, '\nindex:', INDEX)
!python -m rvc_training.cli vocal \
    --lyrics /kaggle/working/outputs/lyrics/song.txt \
    --model {MODEL} --index {INDEX} \
    --output /kaggle/working/outputs/vocals/song_vocal.wav

In [ ]:
import IPython.display as ipd
ipd.Audio('/kaggle/working/outputs/vocals/song_vocal.wav')